# Phase 5 — Operational Forecast Degradation Experiment

**Track B.** Roadmap Phase 5; paper §5.4.

## The question

Almost every published short-term load forecasting result is evaluated on **reanalysis** weather — the weather that actually happened. No operator has that at 20:00 the night before. They have a forecast, and forecasts are wrong in ways that are biased, seasonal, and drifting.

This notebook measures the gap between those two worlds. It is the project's headline operational result, and the reason the paper can state a MAPE that someone could actually reproduce on a real evening.

| | Regime A | Regime B |
|---|---|---|
| Target-day weather | ERA5 reanalysis | archived day-ahead forecast |
| Available at 20:00 the night before? | **No** | Yes |
| What it's good for | comparing feature sets | quoting production accuracy |

Everything else — the model, the features, the weekly refit, the demand lags — is held identical. Only the target-day weather columns change, swapped through `src.contract.swap_weather`.

## Two things that make this trustworthy

1. **Gate G0.** `swap_weather` fed reanalysis reproduces Regime A to `0.000e+00 MW`. So the measured gap is forecast error, not an artifact of the swap. (`python test_contract.py`)
2. **5.2 cross-check.** The archive agrees with forecasts we logged live to 0.36 °C MAE on `t_max`, under the 0.5 °C threshold agreed *before* looking at any demand result.

## Reproducing

```bash
python fetch_weather.py archive-forecast   # one-time, ~30 requests
python run_phase5.py                       # ~20 min, weekly refit, leads 1-3
```

This notebook **reads** `results/operational_gap.json`. It deliberately does not recompute: the paper's tables come from the committed artefact so the notebook and the paper cannot disagree.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from config import RESULTS

gap = json.loads((RESULTS / "operational_gap.json").read_text())
H = gap["H_measured"]
print(f"generated   {gap['generated_at']}")
print(f"model       {gap['model']}")
print(f"H           {H} (measured)   idealised {gap['H_idealised']}")
print(f"test        {gap['test_start']} -> {gap['test_end']}, refit {gap['refit_days']}d")
print(f"leads       {gap['leads']}")
print(f"surge       {gap['surge_definition']}")
print(f"CIs         {gap['ci_method']}")

## 5.2 — Is the archive a valid stand-in for real forecasts?

Our live log starts 2026-08-04: one season, no March–June heat, far too short to backtest. Open-Meteo's Previous Runs API gives archived *past model runs* resolved by lead time back to mid-2021 — 5.4 years.

That substitution only holds if the archive matches what we actually held. If it didn't, every number below would be measuring Open-Meteo's archive rather than our pipeline, and the whole phase would have to be labelled preliminary. The threshold was fixed in advance.

In [ ]:
overlap = pd.read_csv(RESULTS / "archive_vs_log_overlap.csv")
display(overlap.pivot_table(index="column", columns="lead",
                            values=["bias_log_minus_archive", "mae"]).round(3))

g = gap["phase_5_2_archive_vs_log"]
print(f"gate: t_max/cdd_max MAE at lead 1 = {g['gate_metric_mae']:.3f} degC "
      f"(threshold {g['threshold']}) -> {'PASS' if g['passed'] else 'FAIL'}")
print(g["verdict"])

## 5.3 — The dual-regime result

Same model, same walk-forward, only the target-day weather differs.

`H=2` is the headline because it is what the calendar actually permits: Grid-India publishes day *D*'s peak on *D+1*, so an evening forecast for tomorrow is built on demand through **yesterday**. `peak_lag_1` — the single most predictive feature in the dataset — is not available. `H=1` is reported alongside it as the idealised "if demand arrived instantly" bound, and the difference between the two rows is the cost of data latency, which is worth knowing separately from the cost of weather uncertainty.

In [ ]:
def regime_table(subset="overall"):
    rows = []
    for key, v in gap["phase_5_3_regimes"].items():
        if "gap" in key or not isinstance(v, dict) or subset not in v:
            continue
        m = v[subset]
        if not m.get("n"):
            continue
        rows.append({
            "regime": v.get("label", key), "n": m["n"],
            "MAE": round(m["mae"], 1), "MAPE %": round(m["mape"], 2),
            "RMSE": round(m["rmse"], 1), "bias": round(m["bias"], 1),
            "MAE 95% CI": m.get("mae_ci95"), "bias 95% CI": m.get("bias_ci95"),
        })
    return pd.DataFrame(rows)

print("OVERALL"); display(regime_table("overall"))
print("MARCH-JUNE"); display(regime_table("summer_mar_jun"))
print("TOP-5% SURGE DAYS (per calendar year)"); display(regime_table("surge_per_year"))

In [ ]:
head = gap["phase_5_3_regimes"][f"H{H}_gap_lead1"]
rows = [{"subset": k, "n": v["n"], "MAE A": v["mae_a"], "MAE B": v["mae_b"],
         "dMAE": v["d_mae"], "dMAE %": v["d_mae_pct"],
         "bias A": v["bias_a"], "bias B": v["bias_b"], "dbias": v["d_bias"]}
        for k, v in head.items()]
display(pd.DataFrame(rows))

o = head["overall"]
print(f"HEADLINE: {o['mae_a']} -> {o['mae_b']} MW MAE "
      f"({o['d_mae']:+} MW, {o['d_mae_pct']:+}%)")
print(f"          bias {o['bias_a']:+} -> {o['bias_b']:+} MW")

### Reading the surge rows carefully

The top-5% subset is ~15–20 days per calendar year, so **bias matters more than MAE here**. A model that is several hundred MW low on every record-breaking day is operationally dangerous in a way that MAE cannot express — it fails in one direction, exactly when reserve margin is thinnest. That is why D4 fixes the surge threshold as a *per-year quantile* rather than a fixed 19,000 MW line (which would select two days in 2015 and a third of 2026, measuring demand growth instead of model skill), and why every surge figure is reported with a block-bootstrap CI.

## 5.4 — How weather error becomes MW error

In [ ]:
s = gap["phase_5_4_sensitivity"]
print(f"{s['mw_per_degc_t_max']:+.1f} MW per degC of t_max forecast error "
      f"(r={s['corr_t_max_err_vs_mw_shift']:.3f}, n={s['n_days']})")
print(f"{s['mw_per_degc_cdd_max']:+.1f} MW per degC of cdd_max error")
print()
print(f"mean |error| when |t_max err| >  1 degC: {s['mean_abs_err_when_t_max_err_gt_1c']} MW")
print(f"mean |error| when |t_max err| <= 1 degC: {s['mean_abs_err_when_t_max_err_le_1c']} MW")
display(pd.DataFrame(s["worst_days"]))

## 5.5 — Gold-standard check on forecasts we actually logged

5.2 validated the archive's *weather*. This validates the *demand* result on the small set of days where we hold genuinely live forecasts. It is a fidelity check, not a replacement for the headline — the window is one monsoon/NE-monsoon season with no heat in it, which is precisely the regime where the model is least stressed.

In [ ]:
g5 = gap["phase_5_5_real_log"]
if not g5.get("available"):
    print("live log has no usable lead-1 rows yet")
else:
    print(f"window {g5['window'][0]} -> {g5['window'][1]} ({g5['n_days']} days)")
    for name in ("regime_a", "regime_b"):
        m = g5[name]["overall"]
        print(f"  {g5[name]['label']:<34} MAE {m['mae']:7.1f}  "
              f"MAPE {m['mape']:.2f}%  bias {m['bias']:+8.1f}")
    print(f"\n  gap: {g5['gap']['overall']['d_mae']:+} MW MAE")
    print(f"  CAVEAT: {g5['caveat']}")

## What to carry into the paper (§5.4)

Measured on the weekly-refit run, test window 2025-01-01 → 2026-11-10 (n=676):

| | MAE | MAPE | bias |
|---|---|---|---|
| H=2 reanalysis (Regime A) | 483.5 | 2.81% | −3 |
| **H=2 forecast lead 1 (Regime B)** | **525.2** | **3.03%** | −52 |
| H=1 reanalysis *(idealised)* | 399.6 | 2.30% | +3 |
| H=1 forecast lead 1 *(idealised)* | 422.8 | 2.42% | −57 |

1. **Quote the H=2 Regime B figure (525 MW / 3.03%).** The reanalysis number is not reproducible on any real evening. Regime A belongs in feature-selection comparisons, where both sides use it.
2. **Report `H=2` as the headline** and name it honestly: *"day-ahead forecast with one-day demand-data latency."* Note that **data latency costs more than weather uncertainty** — the H=1→H=2 step is ~84 MW against ~42 MW for forecast weather. That is the argument for why `H` deserved measuring.
3. **Report bias beside error on the surge subsets.** On the top 5% (n=35) the gap is 439 → 551 MW MAE with bias −335 → −418 MW. A model consistently several hundred MW low on record days fails in one direction, when reserve margin is thinnest.
4. **The dominant error source is festivals, not weather.** Weather error is +179 MW/°C, but |t_max error| > 1 °C days average 541 MW against 522 MW on calm days. Meanwhile 8 of the 10 worst days are festival-adjacent, over-forecast by 2,300–3,400 MW — three of them the Diwali cluster. `is_holiday` is one binary; Diwali is a multi-day shutdown like Pongal, and only Pongal has a window feature. This is a feature gap for Phase 3.4 to measure.
5. **State that the forecast bias is non-stationary** (+0.49 °C in 2021 → −0.55 °C in 2026). A fixed correction fitted on the full archive would be ~1 °C wrong today, so none is applied. A limitation, not a tuning opportunity.

## Open items

- **The model is Track B's stand-in**, per roadmap C.3 — not Track A's champion. Re-run `python run_phase5.py --champion models/champion_config.json` at Gate G2. The experiment design does not change; only the estimator does.
- Optional Regime A′ (day-0 analysis, splitting "ERA5 vs model analysis" from "lead-time error") and Regime C (retrain on forecasts) were not attempted. Neither changes the headline.
- 5.5's real-log check covers **one monsoon/NE-monsoon season with no heat in it** (99 days). Re-run after summer 2027.
